# Week 2 — Text Sentiment Measures (Pilot)
**Project:** Do LLMs Read 10-Ks Better Than Dictionaries?

This notebook computes three text-based sentiment measures from MD&A sections:

| Track | Method | Tool |
|-------|--------|------|
| 1 | Loughran–McDonald (2011) word counts | `lm_dict` |
| 2 | FinBERT sentence-level neural sentiment | `ProsusAI/finbert` |
| 3 | Llama-3.1-8B-Instruct zero-shot tone score | Midway3 / vLLM |

**Input:** `data_pilot/processed/master_panel_pilot.parquet` (59 firm-years)  
**Output:** `data_pilot/processed/sentiment_scores_pilot.parquet`


In [ ]:
# Install dependencies (run once)
import subprocess, sys

pkgs = ["pyarrow", "tqdm", "requests", "transformers", "torch", "accelerate", "sentencepiece"]
for p in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", p, "-q"], check=False)
print("✓ packages ready")


In [ ]:
import os, re, json, requests, zipfile, warnings
from pathlib import Path
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_DIR = Path(".")
DATA_PILOT  = PROJECT_DIR / "data_pilot"
PROCESSED   = DATA_PILOT  / "processed"
FILINGS_DIR = DATA_PILOT  / "filings"
LLM_BATCH   = DATA_PILOT  / "llm_batches"
LLM_BATCH.mkdir(parents=True, exist_ok=True)

print("Project dir:", PROJECT_DIR.resolve())


In [ ]:
# Load master panel
panel = pd.read_parquet(PROCESSED / "master_panel_pilot.parquet")
print(f"Panel: {panel.shape[0]} firm-years, {panel.shape[1]} columns")
print(panel[["ticker","fyear","mda_len"]].describe())

# ── Read the raw MD&A text for each filing ─────────────────────────────────
def read_mda(row):
    path = PROJECT_DIR / row["mda_path"]
    try:
        return path.read_text(encoding="utf-8", errors="replace")
    except FileNotFoundError:
        return ""

panel["mda_text"] = panel.apply(read_mda, axis=1)
n_empty = (panel["mda_text"].str.len() == 0).sum()
print(f"\n✓ Loaded {len(panel)} MD&A texts  ({n_empty} empty)")
panel[["ticker","fyear","mda_len"]].assign(
    read_len=panel["mda_text"].str.len()
).head(8)


---
## Track 1 — Loughran–McDonald (2011) Dictionary
Downloads the master word list from Bill McDonald's Notre Dame page (one-time).  
Word categories used: **Negative**, **Positive**, **Uncertainty**, **Litigious**, **Strong_Modal**, **Weak_Modal**.

Key metric: `lm_net_sent = (positive − negative) / total_words`


In [ ]:
# ── Download LM Master Dictionary ─────────────────────────────────────────
LM_CSV = DATA_PILOT / "raw" / "LM_MasterDictionary.csv"

if not LM_CSV.exists():
    print("Downloading LM Master Dictionary …")
    url = ("https://sraf.nd.edu/wp-content/uploads/2022/01/"
           "LoughranMcDonald_MasterDictionary_1993-2021.csv")
    r = requests.get(url, timeout=60)
    LM_CSV.write_bytes(r.content)
    print(f"  Saved → {LM_CSV}")
else:
    print(f"Found cached copy: {LM_CSV}")

lm_raw = pd.read_csv(LM_CSV, low_memory=False)
print("LM columns:", lm_raw.columns.tolist()[:15])
print("Rows:", len(lm_raw))


In [ ]:
# ── Build word-set dictionaries ────────────────────────────────────────────
CATS = ["Negative","Positive","Uncertainty","Litigious","Strong_Modal","Weak_Modal"]

# Each column has year of first use (>0) or 0 (never used)
lm_sets = {}
for cat in CATS:
    lm_sets[cat] = set(
        lm_raw.loc[lm_raw[cat] > 0, "Word"].str.upper()
    )
    print(f"  {cat:15s}: {len(lm_sets[cat]):5d} words")

# Flatten to full vocab for fast tokenisation
ALL_LM_WORDS = set().union(*lm_sets.values())
print(f"\n  Total unique LM words: {len(ALL_LM_WORDS)}")


In [ ]:
# ── Tokenise & count ───────────────────────────────────────────────────────
_WORD_RE = re.compile(r"[A-Za-z]+")

def lm_counts(text: str) -> dict:
    tokens = [t.upper() for t in _WORD_RE.findall(text)]
    total  = len(tokens)
    if total == 0:
        return {f"lm_{c.lower()}": 0 for c in CATS} | {
            "lm_total_words": 0, "lm_net_sent": np.nan,
            "lm_tone": np.nan, "lm_uncertainty_pct": np.nan
        }

    counts = {f"lm_{c.lower()}": sum(1 for t in tokens if t in lm_sets[c])
              for c in CATS}
    pos  = counts["lm_positive"]
    neg  = counts["lm_negative"]
    unc  = counts["lm_uncertainty"]

    counts["lm_total_words"]    = total
    counts["lm_net_sent"]       = (pos - neg) / total
    counts["lm_tone"]           = (pos - neg) / (pos + neg) if (pos + neg) > 0 else np.nan
    counts["lm_uncertainty_pct"]= unc / total
    return counts

# Apply to all filings
lm_rows = []
for _, row in tqdm(panel.iterrows(), total=len(panel), desc="LM Dict"):
    lm_rows.append({"gvkey": row["gvkey"], "fyear": row["fyear"],
                    **lm_counts(row["mda_text"])})

lm_df = pd.DataFrame(lm_rows)
print("LM results sample:")
lm_df[["gvkey","fyear","lm_total_words","lm_positive","lm_negative",
       "lm_net_sent","lm_tone","lm_uncertainty_pct"]].head(8)


In [ ]:
# ── Quick sanity plot ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
lm_df["lm_net_sent"].hist(bins=20, ax=axes[0], color="#065A82")
axes[0].set_title("LM Net Sentiment")
lm_df["lm_tone"].hist(bins=20, ax=axes[1], color="#B85042")
axes[1].set_title("LM Tone (±1 scale)")
lm_df["lm_uncertainty_pct"].hist(bins=20, ax=axes[2], color="#2C5F2D")
axes[2].set_title("Uncertainty %")
plt.tight_layout()
plt.savefig(PROCESSED / "lm_distributions.png", dpi=120, bbox_inches="tight")
plt.show()
print("✓ Track 1 complete")


---
## Track 2 — FinBERT Neural Sentiment
Uses `ProsusAI/finbert` (BERT fine-tuned on financial news).

**Strategy:** chunk each MD&A into ≤450-token windows (50-token stride overlap),
score each chunk, then aggregate to a document-level score.

`finbert_net` = mean(P(positive) − P(negative)) across chunks


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

FINBERT_MODEL = "ProsusAI/finbert"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

print("Loading FinBERT tokeniser + model …")
fb_tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
fb_model     = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL)
fb_model.eval().to(DEVICE)
print(f"✓ FinBERT ready  (labels: {fb_model.config.id2label})")


In [ ]:
# ── Chunk text into ≤450-token windows ────────────────────────────────────
CHUNK_TOKENS  = 450   # keep well under 512 limit
STRIDE_TOKENS = 50    # overlap between chunks

def chunk_text(text: str) -> list[str]:
    """Split text into word-level chunks respecting approx token budget."""
    words = text.split()
    # rough approx: 1 word ≈ 1.3 tokens
    step  = int(CHUNK_TOKENS / 1.3)
    stride= int(STRIDE_TOKENS / 1.3)
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i: i + step])
        if chunk.strip():
            chunks.append(chunk)
        i += step - stride
    return chunks if chunks else [text[:2000]]

# ── Score a single batch of chunks ────────────────────────────────────────
@torch.no_grad()
def score_chunks(chunks: list[str]) -> np.ndarray:
    """Returns (N, 3) prob array  [positive, negative, neutral]."""
    BATCH = 16
    probs_all = []
    for i in range(0, len(chunks), BATCH):
        batch = chunks[i: i + BATCH]
        enc   = fb_tokenizer(batch, padding=True, truncation=True,
                             max_length=512, return_tensors="pt").to(DEVICE)
        logits = fb_model(**enc).logits          # (B, 3)
        probs  = F.softmax(logits, dim=-1).cpu().numpy()
        probs_all.append(probs)
    return np.vstack(probs_all)   # (N, 3)

# ── Document-level aggregation ─────────────────────────────────────────────
def finbert_doc_score(text: str) -> dict:
    if not text.strip():
        return {"fb_positive": np.nan, "fb_negative": np.nan,
                "fb_neutral": np.nan, "fb_net": np.nan, "fb_n_chunks": 0}
    chunks = chunk_text(text)
    probs  = score_chunks(chunks)                # (N, 3)
    # FinBERT label order: positive=0, negative=1, neutral=2
    label_map = {v: k for k, v in fb_model.config.id2label.items()}
    pos_idx = label_map.get("positive", 0)
    neg_idx = label_map.get("negative", 1)
    neu_idx = label_map.get("neutral",  2)

    return {
        "fb_positive": float(probs[:, pos_idx].mean()),
        "fb_negative": float(probs[:, neg_idx].mean()),
        "fb_neutral":  float(probs[:, neu_idx].mean()),
        "fb_net":      float((probs[:, pos_idx] - probs[:, neg_idx]).mean()),
        "fb_n_chunks": len(chunks)
    }

# ── Apply to all filings (takes ~2-5 min on CPU, <30s on GPU) ─────────────
fb_rows = []
for _, row in tqdm(panel.iterrows(), total=len(panel), desc="FinBERT"):
    result = finbert_doc_score(row["mda_text"])
    fb_rows.append({"gvkey": row["gvkey"], "fyear": row["fyear"], **result})

fb_df = pd.DataFrame(fb_rows)
print("\nFinBERT results sample:")
fb_df.head(8)


In [ ]:
# ── Diagnostics ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
fb_df["fb_net"].hist(bins=20, ax=axes[0], color="#028090")
axes[0].set_title("FinBERT Net (pos−neg)")
fb_df["fb_positive"].hist(bins=20, ax=axes[1], color="#2C5F2D")
axes[1].set_title("FinBERT P(positive)")
fb_df["fb_n_chunks"].hist(bins=15, ax=axes[2], color="#6D2E46")
axes[2].set_title("Chunks per filing")
plt.tight_layout()
plt.savefig(PROCESSED / "finbert_distributions.png", dpi=120, bbox_inches="tight")
plt.show()
print("✓ Track 2 complete")


---
## Track 3 — Llama-3.1-8B-Instruct (Midway3)

The LLM inference runs on Midway3 GPU nodes via vLLM.  
This section:
1. Builds the **prompt** for each filing
2. Writes sharded **JSONL batch files** ready to upload to Midway3
3. After inference, reads back results and merges them here

### Prompt Design
We ask the model for a JSON object with four scores (0–10):
- `overall_tone`: overall optimism/pessimism of management language
- `forward_looking`: degree of forward-looking positive guidance
- `uncertainty`: expressed uncertainty / hedging
- `risk_disclosure`: severity of risk language


In [ ]:
# ── Prompt template ────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a financial analyst reading a 10-K filing.
Score the Management Discussion and Analysis (MD&A) excerpt on four dimensions.
Respond ONLY with valid JSON, no explanation, no markdown fences.

Output format (integers 0-10):
{"overall_tone": <int>, "forward_looking": <int>, "uncertainty": <int>, "risk_disclosure": <int>}

Scoring guide:
- overall_tone:    0=very negative, 5=neutral, 10=very positive
- forward_looking: 0=backward-looking only, 10=highly forward-looking positive guidance
- uncertainty:     0=highly certain confident language, 10=extreme hedging/uncertainty
- risk_disclosure: 0=no material risks mentioned, 10=severe/numerous risks disclosed
"""

def build_messages(mda_text: str, max_chars: int = 6000) -> list:
    """Truncate to first 6000 chars (captures intro + strategy discussion)."""
    excerpt = mda_text[:max_chars].strip()
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"MD&A excerpt:\n\n{excerpt}"}
    ]

# Preview one prompt
sample = panel.iloc[0]
msgs   = build_messages(sample["mda_text"])
print(f"Filing: {sample['ticker']} FY{sample['fyear']}")
print(f"User message length: {len(msgs[1]['content'])} chars")
print("\n--- System prompt (first 300 chars) ---")
print(msgs[0]["content"][:300])


In [ ]:
# ── Write batch JSONL files ────────────────────────────────────────────────
# Each line: {"id": "...", "ticker": "...", "fyear": ..., "messages": [...]}
# Split into shards of 20 filings for easy SLURM job arrays

SHARD_SIZE = 20
shards = []
records = []

for _, row in panel.iterrows():
    records.append({
        "id":       f"{row['gvkey']}_{row['fyear']}",
        "gvkey":    row["gvkey"],
        "ticker":   row["ticker"],
        "fyear":    int(row["fyear"]),
        "messages": build_messages(row["mda_text"])
    })

# Shard
for i in range(0, len(records), SHARD_SIZE):
    shard_id   = i // SHARD_SIZE
    shard_path = LLM_BATCH / f"batch_{shard_id:03d}.jsonl"
    with open(shard_path, "w") as f:
        for rec in records[i: i + SHARD_SIZE]:
            f.write(json.dumps(rec) + "\n")
    shards.append(str(shard_path))

print(f"✓ Wrote {len(shards)} JSONL shard(s) → {LLM_BATCH}/")
for s in shards:
    n = sum(1 for _ in open(s))
    print(f"   {Path(s).name}: {n} records")


In [ ]:
# ── Verify a batch record ─────────────────────────────────────────────────
with open(LLM_BATCH / "batch_000.jsonl") as f:
    sample_rec = json.loads(f.readline())

print(f"id: {sample_rec['id']}")
print(f"ticker: {sample_rec['ticker']}  fyear: {sample_rec['fyear']}")
print(f"messages: {len(sample_rec['messages'])} turns")
print(f"user text length: {len(sample_rec['messages'][1]['content'])} chars")


---
### 3B. Load Llama Results (run after Midway3 job completes)

After the SLURM job runs, copy the output files back:
```bash
# On your laptop:
scp midway3.rcc.uchicago.edu:/scratch/midway3/$USER/llm_out/*.jsonl \
    "data_pilot/llm_batches/results/"
```
Then run the cell below to parse and merge.


In [ ]:
# ── Parse Llama outputs ────────────────────────────────────────────────────
RESULTS_DIR = LLM_BATCH / "results"

def parse_llm_json(raw: str) -> dict | None:
    """Extract JSON from model output (handles extra whitespace/text)."""
    raw = raw.strip()
    # try direct parse
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    # extract first {...} block
    m = re.search(r"\{[^{}]+\}", raw, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    return None

llm_rows = []
if RESULTS_DIR.exists():
    result_files = sorted(RESULTS_DIR.glob("*.jsonl"))
    print(f"Found {len(result_files)} result file(s)")
    for rf in result_files:
        with open(rf) as f:
            for line in f:
                rec    = json.loads(line)
                parsed = parse_llm_json(rec.get("output", ""))
                if parsed:
                    llm_rows.append({
                        "gvkey":             rec["gvkey"],
                        "fyear":             rec["fyear"],
                        "llm_mgmt_optimism": parsed.get("management_optimism"),
                        "llm_guidance_spec": parsed.get("guidance_specificity"),
                        "llm_unc_hedging":   parsed.get("uncertainty_hedging"),
                        "llm_risk_framing":  parsed.get("risk_framing"),
                        "llm_raw":           rec.get("output","")[:200]
                    })
    llm_df = pd.DataFrame(llm_rows)
    print(f"✓ Parsed {len(llm_df)} Llama scores")
    # normalise to [-1, +1] for net_sentiment equivalent
    for col in ["llm_mgmt_optimism","llm_guidance_spec"]:
        llm_df[f"{col}_norm"] = (llm_df[col] - 5) / 5
    print(llm_df.head())
else:
    print("⚠  Results directory not found yet — run Midway3 job first.")
    print(f"   Expected: {RESULTS_DIR.resolve()}")
    llm_df = pd.DataFrame()


---
## Merge All Tracks → Final Panel


In [ ]:
# ── Merge LM + FinBERT into panel ─────────────────────────────────────────
merged = (
    panel
    .drop(columns=["mda_text"], errors="ignore")   # don't store raw text in parquet
    .merge(lm_df, on=["gvkey","fyear"], how="left")
    .merge(fb_df, on=["gvkey","fyear"], how="left")
)

# Optionally merge Llama scores (only if results file exists)
if not llm_df.empty:
    llm_merge_cols = ["gvkey","fyear"] + [c for c in llm_df.columns
                                           if c not in ("llm_raw",)]
    merged = merged.merge(llm_df[llm_merge_cols], on=["gvkey","fyear"], how="left")

print(f"Merged panel shape: {merged.shape}")
print("Columns:", merged.columns.tolist())


In [ ]:
# ── Correlation table: text measures vs. each other ──────────────────────
TEXT_COLS = ["lm_net_sent","lm_tone","lm_uncertainty_pct",
             "fb_net","fb_positive","fb_negative"]
if not llm_df.empty:
    TEXT_COLS += ["llm_mgmt_optimism_norm","llm_guidance_spec_norm",
                  "llm_unc_hedging","llm_risk_framing"]

corr = merged[TEXT_COLS].corr().round(3)
print("\nCorrelation matrix — text measures:")
print(corr.to_string())


In [ ]:
# ── Cross-sectional summary ───────────────────────────────────────────────
summary_cols = ["ticker","fyear","lm_net_sent","lm_tone",
                "fb_net","lm_total_words"]
print(merged[summary_cols].sort_values(["ticker","fyear"]).to_string(index=False))


In [ ]:
# ── Export ────────────────────────────────────────────────────────────────
OUT_PATH = PROCESSED / "sentiment_scores_pilot.parquet"
merged.to_parquet(OUT_PATH, index=False)
print(f"✓ Saved → {OUT_PATH}")
print(f"  Shape: {merged.shape}")
print(f"  Columns: {merged.columns.tolist()}")

# Also save CSV for quick inspection
merged.to_csv(PROCESSED / "sentiment_scores_pilot.csv", index=False)
print("✓ Also saved CSV")


---
## Preview: Regression Setup (Week 3)

A quick sanity check — do LM and FinBERT measures correlate with future returns
in the expected direction?  Full event-study CAR construction happens in Week 3.


In [ ]:
# ── Preview: add placeholder CAR column (Week 3 will fill this) ──────────
# Load CRSP returns (already pulled in Week 1)
crsp = pd.read_parquet(DATA_PILOT / "raw" / "crsp_pilot.parquet")
print("CRSP columns:", crsp.columns.tolist())
print(crsp.head(3).to_string())


In [ ]:
# ── Sketch of CAR computation (placeholder — full version in week3.ipynb) ─
# For each (permno, earnings_date), compute:
#   CAR[-1,+1] = sum of daily abnormal returns around the EA

from datetime import timedelta

def compute_car(permno, event_date, crsp_df, window=(-1, 1)):
    """Stub — Week 3 will compute proper market-adjusted returns."""
    mask = (
        (crsp_df["permno"] == permno) &
        (crsp_df["date"] >= event_date + timedelta(days=window[0])) &
        (crsp_df["date"] <= event_date + timedelta(days=window[1]))
    )
    sub = crsp_df[mask]
    if sub.empty:
        return np.nan
    # raw return for now (market-adjusted in Week 3)
    return sub["ret"].sum()

print("CAR function defined — full implementation in week3_event_study.ipynb")
print("\nWeek 2 complete ✓")
print("Next: upload JSONL batches to Midway3 → run submit_llama.sh → scp results back")
